## Day08_Anomaly_Detection_and_Warning_Cases.ipynb

# Day 8 — Anomaly Detection and Warning Cases

## Goal

* Identify unusual lice and environmental patterns that may require investigation or early intervention.

In [81]:
# Load libraries
import pandas as pd
from pathlib import Path

In [82]:
feature_file = (
    Path.home()
    / "Documents"
    / "Kazi_Academic"
    / "Projects"
    / "Aquaculture"
    / "fish-health-analytics"
    / "data"
    / "processed"
    / "barentswatch_lice_2025_features.csv"
)

df = pd.read_csv(feature_file)

/tmp/ipykernel_168854/125858662.py:13: DtypeWarning: Columns (0: weekly_lice_limit) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(feature_file)


In [83]:
# Save the data updated dataframe
df.to_csv(feature_file, index=False)

In [84]:
df.shape

(55718, 41)

In [85]:
df.columns.tolist()

['week',
 'year',
 'locality_id',
 'locality_name',
 'adult_female_lice',
 'mobile_lice',
 'sessile_lice',
 'probably_without_fish',
 'lice_counted',
 'municipality_id',
 'municipality',
 'county_id',
 'county',
 'latitude',
 'longitude',
 'weekly_lice_limit',
 'above_weekly_lice_limit',
 'sea_temperature_c',
 'production_area_id',
 'production_area',
 'weekly_lice_limit_numeric',
 'compliance_code',
 'adult_female_lice_lag_1',
 'adult_female_lice_lag_2',
 'adult_female_lice_change',
 'adult_female_lice_change_2w',
 'adult_female_lice_mean_3w',
 'distance_to_limit',
 'ratio_to_limit',
 'previous_breach',
 'future_breach',
 'temperature_flag',
 'sea_temperature_clean',
 'high_temperature_check',
 'temp_previous_week',
 'temp_next_week',
 'temp_neighbor_mean',
 'temp_difference',
 'temperature_lag_1',
 'temperature_change',
 'temperature_mean_3w']

In [86]:
# Understand the distribution of weekly lice changes.
lice_change_summary = df["adult_female_lice_change"].describe()
lice_change_summary

count    28840.000000
mean         0.005415
std          0.253739
min         -6.180000
25%         -0.020000
50%          0.000000
75%          0.050000
max          6.050000
Name: adult_female_lice_change, dtype: float64

In [87]:
# Find the biggest increases:
# Sort rows by weekly lice change

largest_increases = df.sort_values("adult_female_lice_change", ascending=False)

In [88]:
# Choose columns to inspect
columns_to_show = [
    "locality_name",
    "year",
    "week",
    "adult_female_lice",
    "adult_female_lice_lag_1",
    "adult_female_lice_change",
    "ratio_to_limit"
]


In [89]:
# Show the 15 largest weekly increases
largest_increases[columns_to_show].head(15)

,locality_name,year,week,adult_female_lice,adult_female_lice_lag_1,adult_female_lice_change,ratio_to_limit
9230,Klungsholmen,2025,34,8.30,2.25,6.05,16.60
40215,Oksen,2025,40,5.61,0.41,5.20,11.22
16720,Bjørgan,2025,36,5.87,0.96,4.91,11.74
13752,Mjåneset,2025,32,6.53,1.83,4.70,13.06
6562,Geiterøya I,2025,42,4.77,0.67,4.10,9.54
21799,Knappen Solheim,2025,47,4.95,1.12,3.83,9.90
42340,Trettholmosen,2025,33,4.47,0.68,3.79,8.94
16308,Grøttingsøy,2025,40,5.04,1.27,3.77,10.08
38083,Kattholmen II,2025,40,3.56,0.21,3.35,7.12
10840,Haverøy,2025,32,3.63,0.32,3.31,7.26


In [90]:
# Sort from biggest decrease upward
largest_decreases= df.sort_values("adult_female_lice_change", ascending=True)

In [91]:
largest_decreases[columns_to_show].head(15)

,locality_name,year,week,adult_female_lice,adult_female_lice_lag_1,adult_female_lice_change,ratio_to_limit
13753,Mjåneset,2025,33,0.35,6.53,-6.18,0.70
16721,Bjørgan,2025,37,0.40,5.87,-5.47,0.80
39955,Toska N,2025,40,0.28,4.71,-4.43,0.56
9125,Gissøysundet S,2025,33,0.35,4.62,-4.27,0.70
48414,Munkskjæra,2025,36,0.11,4.34,-4.23,0.22
9231,Klungsholmen,2025,35,4.20,8.30,-4.10,8.40
31444,Kalsøyflu,2025,34,0.01,3.64,-3.63,0.02
10746,Fureholmen S,2025,42,0.05,3.45,-3.40,0.10
9234,Klungsholmen,2025,38,0.26,3.64,-3.38,0.52
16310,Grøttingsøy,2025,42,0.28,3.60,-3.32,0.56


In [92]:
# Calculate the 99th percentile of weekly change magnitude
# Take the size of the weekly change

absolute_change = df["adult_female_lice_change"].abs()

In [93]:
# Save the absolute change
df["lice_change_absolute"] = absolute_change

# Find the above which only 1% of change occur

anomaly_threshold = df["lice_change_absolute"].quantile(0.99)

# Show the threshold
anomaly_threshold

np.float64(1.03)

In [94]:
# create the actual anomaly flag.
#  Check whether the absolute change is at least the threshold
is_anomaly = df["lice_change_absolute"] >= anomaly_threshold

In [95]:
#  Save the result as a new column
df["lice_change_anomaly"] = is_anomaly

In [96]:
# Calculate anomaly percentage
anomaly_percent = df["lice_change_anomaly"].mean() * 100

anomaly_percent

np.float64(0.5240676262608134)

In [97]:
# Start by calling every row "Normal"
df["anomaly_type"] = "Normal"

In [98]:
# Mark sudden increases
df.loc[
    df["adult_female_lice_change"] >= anomaly_threshold,
    "anomaly_type"
] = "Sudden increase"

In [99]:
# Mark sudden decreases
df.loc[
    df["adult_female_lice_change"] <= -anomaly_threshold,
    "anomaly_type"
] = "Sudden decrease"

# Are sudden increases followed by future breaches more often than normal weeks?

In [100]:
# Step 1: Keep rows with known future breach status
valid_data = df[df["future_breach"].notna()].copy()

In [101]:
# Step 2: Count future breaches for each anomaly type
anomaly_breach_table = pd.crosstab(
    valid_data["anomaly_type"],
    valid_data["future_breach"]
)

anomaly_breach_table

future_breach,0.0,1.0
anomaly_type,,
Normal,28439,983
Sudden decrease,142,14
Sudden increase,65,57


In [102]:
# Step 3: Calculate percentages within each anomaly group
anomaly_breach_percent = pd.crosstab(
    valid_data["anomaly_type"],
    valid_data["future_breach"],
    normalize="index"
)

anomaly_breach_percent = anomaly_breach_percent * 100

In [103]:
# Step 4: Show the percentages
anomaly_breach_percent.round(2)

future_breach,0.0,1.0
anomaly_type,,
Normal,96.66,3.34
Sudden decrease,91.03,8.97
Sudden increase,53.28,46.72


# test only observations that are currently not breaching

In [104]:
# Keep currently complaint rows
# Keep rows where the current week is **NOT** a breach.

pre_breach_data = valid_data[
    valid_data["compliance_code"] == 0
    ].copy()

In [105]:
# compare anomaly type again
# Compare anomaly type with next week's breach
pre_breach_table = pd.crosstab(
    pre_breach_data["anomaly_type"],
    pre_breach_data["future_breach"],
    normalize="index"
)

In [106]:
# convert to percentages
# Convert proportions to percentages
pre_breach_table = pre_breach_table * 100

In [107]:
# round and display
# Show results
pre_breach_table.round(2)

future_breach,0.0,1.0
anomaly_type,,
Normal,97.80,2.20
Sudden decrease,94.81,5.19


In [108]:
# Count anomaly types among currently compliant observations
pre_breach_data["anomaly_type"].value_counts()

anomaly_type
Normal             27695
Sudden decrease      135
Name: count, dtype: int64

# Build a real early-warning rule instead of an extreme-anomaly rule

In [109]:
# create the warning condition
# Check whether the locality is close to the limit
close_to_limit = df["ratio_to_limit"] >= 0.80 # True when the current lice level has reached at least 80% of the applicable legal limit.

In [110]:
# Step 2: Are lice increasing from the previous week?
lice_increasing = df["adult_female_lice_change"] > 0

In [111]:
# # Check whether the locality is currently NOT in breach
currently_compliant = df["compliance_code"] == 0 

In [112]:
# combine the three conditions
watch_condition = (
    close_to_limit
    & lice_increasing
    & currently_compliant
)


In [113]:
# create a warning column
df["warning_level"] = "Normal"

In [114]:
df.loc[
    watch_condition,
    "warning_level"
] = "Watch"

In [115]:
# count the warnings
df["warning_level"].value_counts()

warning_level
Normal    53592
Watch      2126
Name: count, dtype: int64

In [116]:
# keep rows where next week's outcome is known
# Keep rows where future breach is known
known_future = df["future_breach"].notna()

In [117]:
# keep only currently compliant rows
# Keep rows that are not already in breach
currently_safe = df["compliance_code"] == 0

In [118]:
#  Keep rows suitable for testing our early-warning rule
test_rows = known_future & currently_safe

In [119]:
# Make a dataframe containing only these rows
warning_test = df[test_rows].copy()

In [120]:
# Count future outcomes for Normal and Watch
warning_table = pd.crosstab(
    warning_test["warning_level"],
    warning_test["future_breach"]
)

warning_table

future_breach,0.0,1.0
warning_level,,
Normal,25470,377
Watch,1744,239


In [121]:
#Calculate percentages within each warning group
warning_percent = pd.crosstab(
    warning_test["warning_level"],
    warning_test["future_breach"],
    normalize="index"
)

In [122]:
# Convert proportions to percentages
warning_percent = warning_percent * 100

In [123]:
# Show rounded percentages
warning_percent.round(2)

future_breach,0.0,1.0
warning_level,,
Normal,98.54,1.46
Watch,87.95,12.05


In [124]:
# calculate warning coverage
# Step 1: Keep rows where a breach happened next week
future_breaches = warning_test[
    warning_test["future_breach"] == 1
].copy()

In [125]:
# Step 2: Count how many future breaches were Normal or Watch
breach_warning_counts = future_breaches["warning_level"].value_counts()

breach_warning_counts

warning_level
Normal    377
Watch     239
Name: count, dtype: int64

In [126]:
# Step 3: Calculate percentages
breach_warning_percent = future_breaches[
    "warning_level"
].value_counts(normalize=True) * 100

In [127]:
# Step 4: Show rounded percentages
breach_warning_percent.round(2)

warning_level
Normal    61.2
Watch     38.8
Name: proportion, dtype: float64

In [128]:
# ============================================================
# DAY 8 — FINAL SYNTHESIS
# Goal:
# Combine the warning features we already created,
# rank currently compliant locality-weeks by risk,
# and check whether higher-risk rows have more future breaches.
# ============================================================


# ------------------------------------------------------------
# STEP 1: Keep only rows that are useful for early warning
# ------------------------------------------------------------

# We only want rows that are currently NOT in breach.
currently_compliant = df["compliance_code"] == 0

# We also need to know what happened next week.
future_known = df["future_breach"].notna()

# Keep rows meeting both conditions.
risk_data = df[currently_compliant & future_known].copy()


# ------------------------------------------------------------
# STEP 2: Convert important warning features into percentile ranks
# ------------------------------------------------------------

# Percentile rank means:
# 0.90 = this value is higher than about 90% of observations.
#
# This avoids repeatedly inventing arbitrary thresholds.

# Current pressure relative to the legal limit
risk_data["pressure_rank"] = (
    risk_data["ratio_to_limit"].rank(pct=True)
)

# Recent 2-week lice increase
risk_data["growth_rank"] = (
    risk_data["adult_female_lice_change_2w"].rank(pct=True)
)

# Sustained temperature
risk_data["temperature_rank"] = (
    risk_data["temperature_mean_3w"].rank(pct=True)
)


# ------------------------------------------------------------
# STEP 3: Remove negative growth from the warning score
# ------------------------------------------------------------

# A negative lice change means lice are decreasing.
# We do not want a strong decrease to increase our warning score.

risk_data["growth_rank_positive"] = risk_data["growth_rank"]

decreasing_lice = risk_data["adult_female_lice_change_2w"] <= 0

risk_data.loc[
    decreasing_lice,
    "growth_rank_positive"
] = 0


# ------------------------------------------------------------
# STEP 4: Create a combined risk score
# ------------------------------------------------------------

# We combine:
# 1. How close the locality is to the lice limit
# 2. How strongly lice are increasing
# 3. Whether water has been relatively warm
#
# Each component contributes equally for now.

risk_data["risk_score"] = (
    risk_data["pressure_rank"]
    + risk_data["growth_rank_positive"]
    + risk_data["temperature_rank"]
) / 3


# ------------------------------------------------------------
# STEP 5: Look at the highest-risk locality-weeks
# ------------------------------------------------------------

# Sort from highest risk to lowest risk.
risk_data = risk_data.sort_values(
    "risk_score",
    ascending=False
)

# Select useful columns for investigation.
columns_to_show = [
    "locality_name",
    "year",
    "week",
    "adult_female_lice",
    "adult_female_lice_change",
    "adult_female_lice_change_2w",
    "ratio_to_limit",
    "temperature_mean_3w",
    "risk_score",
    "future_breach"
]

# Show the top 20 highest-risk observations.
top_risk_cases = risk_data[columns_to_show].head(20)

top_risk_cases


# ------------------------------------------------------------
# STEP 6: Divide the risk score into four groups
# ------------------------------------------------------------

# Q1 = lowest-risk 25%
# Q2 = next 25%
# Q3 = next 25%
# Q4 = highest-risk 25%

risk_data["risk_group"] = pd.qcut(
    risk_data["risk_score"],
    q=4,
    labels=["Q1 Low", "Q2", "Q3", "Q4 High"],
    duplicates="drop"
)


# ------------------------------------------------------------
# STEP 7: Calculate future-breach rate for each risk group
# ------------------------------------------------------------

# Group rows by risk level.
risk_groups = risk_data.groupby(
    "risk_group",
    observed=True
)

# Calculate the average of future_breach.
#
# Because:
# 0 = no breach
# 1 = breach
#
# the mean directly gives the proportion of breaches.

breach_rate_by_risk = risk_groups["future_breach"].mean()

# Convert proportion to percentage.
breach_rate_by_risk = breach_rate_by_risk * 100

# Round the result.
breach_rate_by_risk = breach_rate_by_risk.round(2)

# Display.
breach_rate_by_risk


# ------------------------------------------------------------
# STEP 8: Count how many observations are in each risk group
# ------------------------------------------------------------

risk_group_counts = risk_data["risk_group"].value_counts()

risk_group_counts


# ------------------------------------------------------------
# STEP 9: Create investigation profiles
# ------------------------------------------------------------

# Start with a general label.
risk_data["risk_profile"] = "General"


# Profile A:
# Close to the limit and still increasing.
pressure_and_rising = (
    (risk_data["ratio_to_limit"] >= 0.80)
    & (risk_data["adult_female_lice_change"] > 0)
)

risk_data.loc[
    pressure_and_rising,
    "risk_profile"
] = "Close to limit + rising"


# Profile B:
# Not necessarily close to the limit yet,
# but showing strong recent acceleration.

rapid_growth = (
    risk_data["growth_rank_positive"] >= 0.90
)

risk_data.loc[
    rapid_growth,
    "risk_profile"
] = "Rapid lice growth"


# Profile C:
# High pressure combined with sustained relatively warm water.

warm_and_elevated = (
    (risk_data["pressure_rank"] >= 0.75)
    & (risk_data["temperature_rank"] >= 0.75)
)

risk_data.loc[
    warm_and_elevated,
    "risk_profile"
] = "Elevated lice + warm water"


# ------------------------------------------------------------
# STEP 10: Compare future-breach rate by risk profile
# ------------------------------------------------------------

profile_groups = risk_data.groupby(
    "risk_profile"
)

profile_breach_rate = profile_groups[
    "future_breach"
].mean()

profile_breach_rate = profile_breach_rate * 100

profile_breach_rate = profile_breach_rate.round(2)

profile_breach_rate


# ------------------------------------------------------------
# STEP 11: Final Day 8 investigation table
# ------------------------------------------------------------

final_columns = [
    "locality_name",
    "year",
    "week",
    "adult_female_lice",
    "adult_female_lice_change",
    "adult_female_lice_change_2w",
    "ratio_to_limit",
    "temperature_mean_3w",
    "risk_profile",
    "risk_score",
    "future_breach"
]

day8_investigation_table = risk_data[final_columns].head(30)

day8_investigation_table

,locality_name,year,week,adult_female_lice,adult_female_lice_change,adult_female_lice_change_2w,ratio_to_limit,temperature_mean_3w,risk_profile,risk_score,future_breach
51906,Nordfoldleira I,2025,32,0.48,0.37,0.44,0.96,19.286667,Elevated lice + warm water,0.995614,0.0
14377,Ebne,2025,33,0.48,0.25,0.40,0.96,18.600000,Elevated lice + warm water,0.994388,0.0
24591,Litle Lunnøy,2025,31,0.47,-1.27,0.43,0.94,17.100000,Elevated lice + warm water,0.989482,0.0
14533,Maradalen,2025,33,0.47,0.03,0.42,0.94,17.133333,Elevated lice + warm water,0.989345,0.0
11204,Rotøy,2025,32,0.48,0.42,0.46,0.96,16.103333,Elevated lice + warm water,0.989193,0.0
52639,Skarvhammaren,2025,37,0.47,0.27,0.43,0.94,16.933333,Elevated lice + warm water,0.989177,0.0
51855,Sandskjæret,2025,33,0.47,0.41,0.43,0.94,16.926667,Elevated lice + warm water,0.989151,0.0
15310,Gudmundset,2025,30,0.47,0.12,0.42,0.94,16.666667,Elevated lice + warm water,0.988078,0.0
9753,Naveide,2025,37,0.49,0.30,0.41,0.98,15.576667,Elevated lice + warm water,0.985350,0.0
14378,Ebne,2025,34,0.49,0.01,0.26,0.98,17.266667,Elevated lice + warm water,0.985147,0.0


In [130]:
# Show breach rate for each risk quartile
breach_rate_by_risk

risk_group
Q1 Low     0.08
Q2         0.57
Q3         2.28
Q4 High    6.41
Name: future_breach, dtype: float64

In [131]:
# Show number of observations in each risk group
risk_group_counts

risk_group
Q1 Low     6361
Q2         6361
Q4 High    6361
Q3         6360
Name: count, dtype: int64

In [132]:
# Save the updated feature dataset
df.to_csv(feature_file, index=False)

In [133]:
# Save the Day 8 risk-analysis dataset
risk_file = feature_file.parent / "day08_risk_analysis.csv"

risk_data.to_csv(risk_file, index=False)